In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torchvision 
import numpy as np 
import matplotlib.pyplot as plt 
import cv2 



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hsankesara/flickr-image-dataset")

!mkdir data
!mv /home/$USER/.cache/kagglehub/datasets/hsankesara/flickr-image-dataset data/


Resuming download from 560988160 bytes (8204408358 bytes left)...
Resuming download to /home/kalzemic/.cache/kagglehub/datasets/hsankesara/flickr-image-dataset/1.archive (560988160/8765396518) bytes left.


100%|██████████| 8.16G/8.16G [06:05<00:00, 22.4MB/s] 

Extracting files...


/data /home/kalzemic/.cache/kagglehub/datasets/hsankesara/flickr-image-dataset/versions/1


In [ ]:
%%bash

mkdir -p data/images data/labels

mkdir -p data/images/train data/images/val data/labels/train data/labels/val


## Construct a Prompts file to later be fed to Sam3 for Label Production

In [ ]:
import pandas as pd
import os
import re

results_path = os.path.join(os.getcwd(),'data','flickr-image-dataset','versions','1','flickr30k_images','results.csv')

results_df = pd.read_csv(results_path, sep='|')
results_df[' comment'] = results_df[' comment'].fillna('')

person_keywords = ['mans?', 'woman', 'women', 'persons?', 'child', 'children',
                   'men', 'peoples?', 'people', 'boys?', 'girls?', 'kids?',
                   'guys?', 'ladys?', 'ladies', 'adults?', 'babies', 'baby']
vehicle_keywords = ['cars?', 'trucks?', 'bus', 'buses', 'motorcycles?',
                    'bikes?', 'bicycles?', 'vans?', 'scooters?',
                    'tractors?', 'vehicles?', 'taxis?', 'jeeps?']

person_regex  = r'\b(?:' + '|'.join(person_keywords)  + r')\b'
vehicle_regex = r'\b(?:' + '|'.join(vehicle_keywords) + r')\b'

results_df['has_person']  = results_df[' comment'].str.contains(person_regex,  flags=re.IGNORECASE, regex=True)
results_df['has_vehicle'] = results_df[' comment'].str.contains(vehicle_regex, flags=re.IGNORECASE, regex=True)

results_df = results_df.groupby('image_name').agg({
    'has_person':  'any',
    'has_vehicle': 'any',
}).reset_index()

results_df['persons']  = results_df['has_person'].map(lambda b: 'person' if b else '')
results_df['vehicles'] = results_df['has_vehicle'].map(lambda b: 'vehicle' if b else '')

filtered_df = results_df[['image_name', 'persons', 'vehicles']]
filtered_df = filtered_df[
    filtered_df['persons'].str.len().gt(0) |
    filtered_df['vehicles'].str.len().gt(0)
]
filtered_df.to_csv(os.path.join(os.getcwd(),'data','prompts','prompts.csv'), index=False)

## Initialize Sam3 and run inference to produce labels

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

sam = build_sam3_image_model()
sam_processor = Sam3Processor(sam)

In [ ]:
import torch
import pandas as pd
import os
from PIL import Image

import cv2, numpy as np
SLICE = 100

ONE_HOT = {'person':0,'vehicle':1}
TARGET_PER_CLASS = 800 

image_path = os.path.join(os.getcwd(), "data", 'flickr-image-dataset', 'versions', '1', 'flickr30k_images', 'flickr30k_images')
label_path = os.path.join(os.getcwd(), 'data', 'labels')
os.makedirs(label_path, exist_ok=True)

prompts = pd.read_csv('data/prompts/prompts.csv')

# Categorize rows
prompts['has_person']  = prompts['persons'].notna()
prompts['has_vehicle'] = prompts['vehicles'].notna()

both_df    = prompts[prompts['has_person'] & prompts['has_vehicle']]
vehicle_df = prompts[prompts['has_vehicle'] & ~prompts['has_person']]
person_df  = prompts[prompts['has_person']  & ~prompts['has_vehicle']]

print(f"Pool: both={len(both_df)}, vehicle-only={len(vehicle_df)}, person-only={len(person_df)}")

# Build a balanced work list
# - All "both" first (each contributes to BOTH class counts)
# - Then top up vehicle-only and person-only to reach TARGET_PER_CLASS each
both_rows    = both_df.sample(frac=1, random_state=42)
vehicle_rows = vehicle_df.sample(frac=1, random_state=42)
person_rows  = person_df.sample(frac=1, random_state=42)

# After taking all "both", how many more single-class images do we need?
needed_vehicle = max(0, TARGET_PER_CLASS - len(both_rows))
needed_person  = max(0, TARGET_PER_CLASS - len(both_rows))

selected = pd.concat([
    both_rows,
    vehicle_rows.head(needed_vehicle),
    person_rows.head(needed_person),
])
selected = selected.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Processing {len(selected)} images")


def run_sam3_for_prompt(img, prompt):
    """Returns list of (x1, y1, x2, y2) boxes for the given text prompt."""
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        state = sam_processor.set_image(img)
        result = sam_processor.set_text_prompt(state=state, prompt=prompt)
    return result['boxes'].cpu().numpy()


def yolo_line(class_id, x1, y1, x2, y2, img_w, img_h):
    x1 = max(0, min(x1, img_w))
    x2 = max(0, min(x2, img_w))
    y1 = max(0, min(y1, img_h))
    y2 = max(0, min(y2, img_h))
    w = (x2 - x1) / img_w
    h = (y2 - y1) / img_h
    if w <= 0 or h <= 0:
        return None
    x_center = ((x1 + x2) / 2) / img_w
    y_center = ((y1 + y2) / 2) / img_h
    return f'{ONE_HOT[class_id]} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n'


for row in selected.itertuples():
    img_filepath = os.path.join(image_path, row.image_name)
    if not os.path.exists(img_filepath):
        print(f'image missing: {row.image_name}, skipping')
        continue

    img = Image.open(img_filepath)
    img_w, img_h = img.size

    all_lines = []

    # Run SAM3 once per class that's present in this image
    if row.has_person:
        print(f'SAM3 person inference on {row.image_name}')
        boxes = run_sam3_for_prompt(img, row.persons)
        for box in boxes:
            x1, y1, x2, y2 = box.astype(int)
            line = yolo_line('person', x1, y1, x2, y2, img_w, img_h)
            if line:
                all_lines.append(line)

    if row.has_vehicle:
        print(f'SAM3 vehicle inference on {row.image_name}')
        boxes = run_sam3_for_prompt(img, row.vehicles)
        for box in boxes:
            x1, y1, x2, y2 = box.astype(int)
            line = yolo_line('vehicle', x1, y1, x2, y2, img_w, img_h)
            if line:
                all_lines.append(line)

    if not all_lines:
        print(f'no boxes found for {row.image_name}, skipping')
        continue

    label_filename = os.path.join(label_path, row.image_name.replace('.jpg', '.txt'))
    with open(label_filename, 'w') as fout:
        fout.writelines(all_lines)




: 

## Split to train and val directories for the YOLO format
 

In [ ]:
import shutil, os, random
from pathlib import Path
from collections import defaultdict

random.seed(42)

image_src = os.path.join(os.getcwd(), "data", 'flickr-image-dataset', 'versions', '1', 'flickr30k_images', 'flickr30k_images')
label_src = os.path.join(os.getcwd(), 'data', 'labels')

# Targets (per project requirements: 500 train + 100 val per class)
TRAIN_PER_CLASS = 500
VAL_PER_CLASS = 100

# Categorize each labeled image by what it contains
person_only = []      # has person but no vehicle
vehicle_only = []     # has vehicle but no person
both = []             # has both

for label_file in Path(label_src).glob("*.txt"):
    classes_in_file = set()
    with open(label_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cls_id = int(line.split()[0])
            classes_in_file.add(cls_id)

    if not classes_in_file:
        continue  # empty label file
    if 0 in classes_in_file and 1 in classes_in_file:
        both.append(label_file.stem)
    elif 0 in classes_in_file:
        person_only.append(label_file.stem)
    elif 1 in classes_in_file:
        vehicle_only.append(label_file.stem)

print(f"Pool sizes — person-only: {len(person_only)}, vehicle-only: {len(vehicle_only)}, both: {len(both)}")

random.shuffle(person_only)
random.shuffle(vehicle_only)
random.shuffle(both)

# Strategy:
#   - Use ALL "both" images for free coverage of both classes
#   - Top up with person-only and vehicle-only to hit per-class targets
#
# Count current per-class coverage from "both":
person_count = len(both)    # each "both" image contributes 1 to person count
vehicle_count = len(both)   # and 1 to vehicle count

# How many more we need of each (after "both")
person_need = max(0, (TRAIN_PER_CLASS + VAL_PER_CLASS) - person_count)
vehicle_need = max(0, (TRAIN_PER_CLASS + VAL_PER_CLASS) - vehicle_count)

if len(vehicle_only) < vehicle_need:
    print(f"WARNING: only {len(vehicle_only)} vehicle-only images, need {vehicle_need}. Will use all available.")
    vehicle_need = len(vehicle_only)
if len(person_only) < person_need:
    print(f"WARNING: only {len(person_only)} person-only images, need {person_need}.")
    person_need = len(person_only)

# Build the combined pool
selected = both + vehicle_only[:vehicle_need] + person_only[:person_need]
random.shuffle(selected)

# Split: take TRAIN_PER_CLASS + VAL_PER_CLASS images roughly proportionally
# Simpler: split the selected pool 5:1 (matches 500:100 ratio)
n_val = (len(selected) * VAL_PER_CLASS) // (TRAIN_PER_CLASS + VAL_PER_CLASS)
val_stems = selected[:n_val]
train_stems = selected[n_val:]

# Prepare directories
for split in ["train", "val"]:
    for kind in ["images", "labels"]:
        d = f"data/{kind}/{split}"
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d)

def copy_split(stems, split):
    for stem in stems:
        shutil.copy2(os.path.join(image_src, f"{stem}.jpg"),
                     os.path.join("data", "images", split, f"{stem}.jpg"))
        shutil.copy2(os.path.join(label_src, f"{stem}.txt"),
                     os.path.join("data", "labels", split, f"{stem}.txt"))

copy_split(train_stems, "train")
copy_split(val_stems, "val")

# Report final class balance
def count_classes(split):
    p, v = 0, 0
    for f in Path(f"data/labels/{split}").glob("*.txt"):
        with open(f) as fh:
            classes = {int(l.split()[0]) for l in fh if l.strip()}
        if 0 in classes: p += 1
        if 1 in classes: v += 1
    return p, v

train_p, train_v = count_classes("train")
val_p, val_v = count_classes("val")
print(f"train: {len(train_stems)} images  |  person: {train_p}, vehicle: {train_v}")
print(f"val:   {len(val_stems)} images  |  person: {val_p}, vehicle: {val_v}")

## Custom Dataset class

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A

class DetectionDataset(Dataset):


    def __init__(self, images, labels, img_size=224, transform=None):

        self.images = images
        self.labels = labels
        self.img_size = img_size
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.images[index]
        label = self.labels[index]

        if self.transform is not None:
            if len(label) > 0:
                boxes = label[:, 1:].tolist()
                class_labels = label[:, 0].tolist()
            else:
                boxes = []
                class_labels = []
            out = self.transform(image=image, bboxes=boxes, class_labels=class_labels)
            image = out['image']
            if out['bboxes']:
                label = torch.tensor(
                    [[c, *b] for c, b in zip(out['class_labels'], out['bboxes'])],
                    dtype=torch.float32
                )
            else:
                label = torch.zeros((0, 5), dtype=torch.float32)

        image = torch.from_numpy(image).permute(2, 0, 1).float()

        
        if len(label) > 0:
            cls = label[:, 0].long()            
            cx = label[:, 1] * self.img_size
            cy = label[:, 2] * self.img_size
            w = label[:, 3] * self.img_size
            h = label[:, 4] * self.img_size
            boxes = torch.stack([
                cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2
            ], dim=1).clamp(min=0, max=self.img_size)
        else:
            cls = torch.zeros((0,), dtype=torch.int64)
            boxes = torch.zeros((0, 4), dtype=torch.float32)

        target = {
            'boxes': boxes,
            'labels': cls,
        }
        return image, target


def Collate_fn(batch):

    imgs, targets = zip(*batch)
    imgs = torch.stack(imgs)
    return imgs, list(targets)



def getTorchLoaders(train_data, val_data, train_labels, val_labels, batchsize):
    
    train_transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Affine(translate_percent=0.1, scale=(0.7, 1.3), rotate=0, p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3, clip=True))

    val_transform = A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3, clip=True))
    train_dataset = DetectionDataset(
    	train_data,
    	[torch.tensor(lbl, dtype=torch.float32) for lbl in train_labels],
    	img_size=224,
    	transform=train_transform,
    )
    val_dataset = DetectionDataset(
    	val_data,
    	[torch.tensor(lbl, dtype=torch.float32) for lbl in val_labels],
    	img_size=224,
    	transform=val_transform,  
    )
    train_loader = DataLoader(train_dataset,batch_size=batchsize,shuffle=True, drop_last=True, collate_fn=Collate_fn)
    val_loader = DataLoader(val_dataset,batch_size=batchsize, shuffle=False, collate_fn=Collate_fn)

    return train_loader, val_loader



In [ ]:
import cv2
import numpy as np

def getData(dir_path='.'):
    train_data_path = f'{dir_path}/images/train'
    train_labels_path = f'{dir_path}/labels/train'
    val_data_path = f'{dir_path}/images/val'
    val_labels_path = f'{dir_path}/labels/val'

    train_data = []
    val_data = []
    train_labels = []
    val_labels = []

    label_files = sorted(os.listdir(train_labels_path))

    for idx, label_file in enumerate(label_files):
        

        base_name= os.path.splitext(label_file)[0]
        image_file = base_name + '.jpg'

        
        label_path = os.path.join(train_labels_path, label_file)
        label = np.loadtxt(open(label_path, 'rb'), dtype=np.float32)
        if label.ndim == 1:
            label = np.expand_dims(label, axis=0)
        
        
        img_path = os.path.join(train_data_path, image_file)
        if not os.path.exists(img_path):
            print(f"Warning: Image {image_file} not found for label {label_file}. Skipping.")
            continue 

        img = cv2.imread(img_path)
        img = cv2.cvtColor(cv2.resize(img,(224,224)),cv2.COLOR_BGR2RGB)
        
        train_data.append(img)
        train_labels.append(label)

    label_files = sorted(os.listdir(val_labels_path))

    for idx, label_file in enumerate(label_files):
        if idx > 1000: break 

        base_name= os.path.splitext(label_file)[0]
        image_file = base_name + '.jpg'


        label_path = os.path.join(val_labels_path, label_file)
        label = np.loadtxt(open(label_path, 'rb'), dtype=np.float32)
        if label.ndim == 1:
            label = np.expand_dims(label, axis=0)
        
        
        img_path = os.path.join(val_data_path, image_file)
        if not os.path.exists(img_path):
            print(f"Warning: Image {image_file} not found for label {label_file}. Skipping.")
            continue 

        img = cv2.imread(img_path)
        img = cv2.cvtColor(cv2.resize(img,(224,224)),cv2.COLOR_BGR2RGB)
        
        val_data.append(img)
        val_labels.append(label)
    
    
    return np.stack(train_data), np.stack(val_data), train_labels, val_labels

## EdgeVision Neural Network Module

## EdgeVision  Karpathy Sanity Check

Create Files to Overfit

In [5]:
%%bash

mkdir -p data/overfit/images/train
mkdir -p data/overfit/labels/train

find data/images/train -type f | head -30 | while read img; do
    base=$(basename "$img")
    name="${base%.*}"

    cp "$img" data/overfit/images/train/
    cp "data/labels/train/${name}.txt" data/overfit/labels/train/
done

Run Karpathy overfit test on YOLO implementation

In [6]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="overfit.yaml",
    epochs=300,
    imgsz=800,
    batch=4,
    device=0,

    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    fliplr=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    erasing=0.0,

    patience=300,
    workers=0,
    plots=True,

    project="runs/debug",
    name="EdgeVision_overfit_test",
)

Ultralytics 8.4.76 🚀 Python-3.12.13 torch-2.12.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060, 7805MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=overfit.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=EdgeVision_overfit_test-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, p

Train Run

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt') 
results = model.train(
    data='data.yaml',
    epochs=60,
    imgsz=800,
    batch=16,
    device=0,        # GPU
    project='runs/edge_detector',
    name='EdgeVision_800',
)

print(results.box.map)      
print(results.box.map50)     
print(results.box.mp)        
print(results.box.mr)      


Ultralytics 8.4.76 🚀 Python-3.12.13 torch-2.12.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060, 7805MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=EdgeVision_800-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/edge_detector/EdgeVision_800/weights/best.pt")
metrics = model.val(data="data/data.yaml", imgsz=800, plots=True)

print(metrics.box.map)
print(metrics.box.map50)
print(metrics.box.mp)
print(metrics.box.mr)